In [4]:
!git clone https://github.com/Yoki-SyZhang/RAG-based-Interactive-AI-for-MSADS-Website.git


%cd RAG-based-Interactive-AI-for-MSADS-Website/

import os
data_path = "data/cleaned_sections/"
print(os.listdir(data_path))

Cloning into 'RAG-based-Interactive-AI-for-MSADS-Website'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (314/314), done.
remote: Compressing objects: 100% (251/251), done.
remote: Total 314 (delta 73), reused 284 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (314/314), 1.35 MiB | 8.58 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/RAG-based-Interactive-AI-for-MSADS-Website/RAG-based-Interactive-AI-for-MSADS-Website
['page_159_cleaned.json', 'page_147_cleaned.json', 'page_165_cleaned.json', 'page_146_cleaned.json', 'page_042_cleaned.json', 'page_163_cleaned.json', 'page_161_cleaned.json', 'page_148_cleaned.json', 'page_164_cleaned.json', 'page_166_cleaned.json', 'page_145_cleaned.json', 'page_162_cleaned.json', 'page_149_cleaned.json', 'page_160_cleaned.json']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
db_path = '/content/drive/MyDrive/vector_store'
os.makedirs(db_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import json

file_path = "data/cleaned_sections/page_042_cleaned.json"

with open(file_path, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

formatted_json = json.dumps(sample_data, indent=4, ensure_ascii=False)
print(formatted_json[:1500])

{
    "source_file": "page_042.json",
    "url": [
        "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/"
    ],
    "canonical_url": "https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/",
    "page_title": "Master's in Applied Data Science | DSI",
    "meta_description": "Explore how UChicago's data science master's degree can develop you into a leader in the field by elevating your technical skillset.",
    "breadcrumbs": "Home > Education > Masters Programs > Ms In Applied Data Science",
    "scraped_at": "2026-04-19T01:41:58.015608+00:00",
    "structured_sections": [
        {
            "section_index": 0,
            "text_markdown": "The University of Chicago’s MS in Applied Data Science program equips you with in-demand expertise and an unparalleled network of global alumni. Take the next step and start your application today."
        },
        {
            "section_index": 1,
            "text_

In [14]:
# !pip install -q langchain langchain-community chromadb sentence-transformers langchain-core langchain-text-splitters
# Chunking
import json
import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

data_folder = "data/cleaned_sections/*.json"
raw_documents = []
file_paths = glob.glob(data_folder)

print(f"Reading {len(file_paths)} files")

for file_path in file_paths:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

        url = data.get("canonical_url", "")
        title = data.get("page_title", "")
        breadcrumbs = data.get("breadcrumbs", "")

        sections = data.get("structured_sections", [])
        for sec in sections:
            text_content = sec.get("text_markdown", "")

            if not text_content.strip():
                continue

            doc = Document(
                page_content=text_content,
                metadata={
                    "url": url,
                    "title": title,
                    "breadcrumbs": breadcrumbs,
                    "section_index": sec.get("section_index", 0)
                }
            )
            raw_documents.append(doc)

print(f"Getting {len(raw_documents)} raw documents")

Reading 14 files
Getting 158 raw documents


In [15]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,\
    chunk_overlap=50,\
    separators=["\n## ", "\n\n", "\n", ".", " ", ""]\
)

final_chunks = text_splitter.split_documents(raw_documents)
print(f"Final chunks for {len(final_chunks)}")

Final chunks for 959


In [16]:
# embedding
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vector_db = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory=db_path
)

print(f"embedding finished and the database is stored at {db_path}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding finished and the database is stored at /content/drive/MyDrive/MSADS_RAG_DB


In [17]:
retriever = vector_db.as_retriever(search_kwargs={"k": 2})
test_query = "What is the application deadline?"
results = retriever.invoke(test_query)

for i, doc in enumerate(results):
    print(f"Result: {i+1}]")
    print(f"Location: {doc.metadata.get('breadcrumbs')}")
    print(f"url: {doc.metadata.get('url')}")
    print(f"Content: {doc.page_content}")

Result: 1]
Location: Home > Education > Masters Programs > Ms In Applied Data Science
url: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/
Content: ## Start Your Application
The applicationto be considered for Autumn 2026 enrollment is now open! The final application deadline for full- and part-time entrance in Autumn 2026 is June 23, 2026.
The In-Person Program and the Online Program admit full- and part-time students for entrance in the autumn quarter.
Result: 2]
Location: Home > Education > Masters Programs > Ms In Applied Data Science
url: https://datascience.uchicago.edu/education/masters-programs/ms-in-applied-data-science/
Content: ## Start Your Application
The applicationto be considered for Autumn 2026 enrollment is now open! The final application deadline for full- and part-time entrance in Autumn 2026 is June 23, 2026.
The In-Person Program and the Online Program admit full- and part-time students for entrance in the autumn quarter.
